# RTC Learning / Competition Behavior Analysis

Pickle-first notebook for the revised RTC figures. It follows the style and colors from `reward_summary_figure_adjustable.ipynb` / `make_reward_summary_figure.py` and separates learning dynamics from reward-competition behavior.

In [ ]:
%matplotlib inline
# %matplotlib widget
import sys
from importlib import reload
from pathlib import Path

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == "Reward_Training-Competition" else Path.cwd() / "Reward_Training-Competition"
PROJECT_ROOT = NOTEBOOK_DIR.parent
for path in (PROJECT_ROOT, NOTEBOOK_DIR, NOTEBOOK_DIR / "Reward_Training", NOTEBOOK_DIR / "Reward_Competition"):
    sys.path.insert(0, str(path))

from figure_settings import apply_plot_style, save_figure
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind

import make_reward_summary_figure as rsf
reload(rsf)

STYLE = apply_plot_style(font_size=8, panel_width=3.0, panel_height=2.2)
rsf.apply_reward_summary_style()
REGIONS = ("NAc", "mPFC")
DAYS = ("Day 1", "Day 10")
output_dir = NOTEBOOK_DIR / "Reward_Summary_Figures" / "rtc_revised"
output_dir.mkdir(parents=True, exist_ok=True)
save_formats = ("png", "svg", "pdf")
print(output_dir)

## Settings

In [ ]:
rt_pickle_paths = {
    ("Day 1", "NAc"): [NOTEBOOK_DIR / "preprocessed_reward_training_day1_nac.pkl"],
    ("Day 10", "NAc"): [NOTEBOOK_DIR / "preprocessed_reward_training_day10_nac.pkl"],
    ("Day 1", "mPFC"): [NOTEBOOK_DIR / "preprocessed_reward_training_day1_mpfc.pkl"],
    ("Day 10", "mPFC"): [NOTEBOOK_DIR / "preprocessed_reward_training_day10_mpfc.pkl"],
}
rc_pickle_paths = [NOTEBOOK_DIR / "Reward_Competition" / "preprocessed_reward_competition.pkl"]

learning_tone_window = (-4, 20)
learning_pe_window = (-4, 20)
reward_pickup_deadline_s = 20
tone_metric_col = "Tone Mean Z-score"
pe_metric_col = "PE Mean Z-score"

# Exclude no-pickup trials for day 1 vs day 10 statistical bars.
require_reward_pickup = True

# Choose example animals after loading, or leave None to use the first available animal per region.
example_subjects = {"NAc": None, "mPFC": None}

# Optional folder containing full-session BORIS competition-bout CSVs. If it exists, duration/ITI plots use it.
competition_bout_csv_folder = NOTEBOOK_DIR / "Reward_Competition" / "full_data_comp_bout_csvs"

## Load Pickles

In [ ]:
rt_exps, rc_exp = rsf.load_summary_pickles(rt_pickle_paths=rt_pickle_paths, rc_pickle_paths=rc_pickle_paths)
print("Loaded reward training and reward competition pickles.")
print({key: exp.da_df.shape for key, exp in rt_exps.items()})
print("RC da_df", rc_exp.da_df.shape)

## Helpers

In [ ]:
def as_list(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (list, tuple)):
        return list(value)
    return []

def shared_session_id(file_name):
    parts = str(file_name).split("-", 1)
    return parts[1] if len(parts) > 1 else str(file_name)

def first_matching_col(df, names):
    for name in names:
        if name in df.columns:
            return name
    lowered = {c.lower(): c for c in df.columns}
    for name in names:
        if name.lower() in lowered:
            return lowered[name.lower()]
    return None

def trial_rows_from_df(df, day=None, region=None):
    rows = []
    for _, row in df.iterrows():
        subject = str(row.get("subject_name", ""))
        cues = as_list(row.get("filtered_sound_cues", row.get("sound cues onset", [])))
        pes = as_list(row.get("filtered_port_entries", row.get("port entries onset", [])))
        tone_metric = as_list(row.get(tone_metric_col, []))
        pe_metric = as_list(row.get(pe_metric_col, []))
        n = max(len(cues), len(tone_metric), len(pe_metric))
        for i in range(n):
            cue = cues[i] if i < len(cues) else np.nan
            pe = pes[i] if i < len(pes) else np.nan
            latency = pe - cue if np.isfinite(cue) and np.isfinite(pe) else np.nan
            rows.append({
                "day": day,
                "region": region,
                "subject_name": subject,
                "file name": row.get("file name", ""),
                "session_id": shared_session_id(row.get("file name", "")),
                "trial_idx": i,
                "tone_time_s": cue,
                "port_entry_time_s": pe,
                "reward_latency_s": latency,
                "reward_by_20s": np.isfinite(latency) and (0 <= latency <= reward_pickup_deadline_s),
                tone_metric_col: tone_metric[i] if i < len(tone_metric) else np.nan,
                pe_metric_col: pe_metric[i] if i < len(pe_metric) else np.nan,
            })
    return pd.DataFrame(rows)

def learning_trials_table(rt_exps):
    return pd.concat(
        [trial_rows_from_df(exp.da_df, day=day, region=region) for (day, region), exp in rt_exps.items()],
        ignore_index=True,
    )

def subject_means(df, value_col, group_cols):
    out = df.copy()
    out[value_col] = pd.to_numeric(out[value_col], errors="coerce")
    return out.groupby(group_cols + ["subject_name"], as_index=False)[value_col].mean()

def welch_rows(group_df, value_col, label_col, pairs):
    rows = []
    for a, b in pairs:
        av = pd.to_numeric(group_df.loc[group_df[label_col] == a, value_col], errors="coerce").dropna().to_numpy()
        bv = pd.to_numeric(group_df.loc[group_df[label_col] == b, value_col], errors="coerce").dropna().to_numpy()
        t, p = ttest_ind(av, bv, equal_var=False, nan_policy="omit")
        rows.append({"group_a": a, "group_b": b, "n_a": len(av), "n_b": len(bv), "mean_a": np.nanmean(av) if len(av) else np.nan, "mean_b": np.nanmean(bv) if len(bv) else np.nan, "t": t, "p": p})
    return pd.DataFrame(rows)

def save_all(fig, base_name):
    saved = []
    for fmt in save_formats:
        path = output_dir / f"{base_name}.{fmt}"
        fig.savefig(path, dpi=300, bbox_inches="tight", transparent=True)
        saved.append(path)
    return saved

# Learning Data

In [ ]:
learning_trials = learning_trials_table(rt_exps)
learning_trials.head()

## Percent Reward Picked Up By 20 s: Day 1 vs Day 10

In [ ]:
pickup_subject = (
    learning_trials.groupby(["region", "day", "subject_name"], as_index=False)
    .agg(percent_reward_by_20s=("reward_by_20s", lambda x: 100 * np.nanmean(x.astype(float))), n_trials=("reward_by_20s", "size"))
)
pickup_stats = []
fig, axes = plt.subplots(1, 2, figsize=(5.6, 2.25), sharey=True)
for ax, region in zip(axes, REGIONS):
    groups = [
        {"label": day, "values": pickup_subject.query("region == @region and day == @day")["percent_reward_by_20s"].to_numpy(), "color": rsf.DAY_COLORS[region][day]}
        for day in DAYS
    ]
    stats = rsf.plot_bar_groups(ax, groups, "% trials reward by 20 s", region, comparisons=[(0, 1)])
    stats.insert(0, "region", region)
    pickup_stats.append(stats)
fig.suptitle("Alone training behavior", y=1.05)
plt.show()
save_all(fig, "learning_reward_pickup_by_20s")
pickup_stats = pd.concat(pickup_stats, ignore_index=True)
pickup_stats

## Learning PSTHs To 20 s

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(7.0, 4.7), sharex=False, sharey=False)
for r, region in enumerate(REGIONS):
    groups = [(day, rt_exps[(day, region)].da_df, None, rsf.DAY_COLORS[region][day]) for day in DAYS]
    rsf.plot_psth_groups(axes[r, 0], groups, "Tone", region, learning_tone_window, f"{region} tone aligned", reward_line=True)
    rsf.plot_psth_groups(axes[r, 1], groups, "Pre_PE", region, learning_pe_window, f"{region} port-entry aligned", reward_line=False)
fig.tight_layout()
plt.show()
save_all(fig, "learning_psths_to_20s")

## Trial-Average Heatmaps To 20 s

In [ ]:
def event_matrix_by_trial(df, region, event_type="Tone", time_window=(-4, 20), selector=None):
    records = rsf.collect_event_records(df, event_type, region, selector=selector)
    common_t = rsf.make_common_time(records, time_window)
    if common_t is None:
        return None, None
    traces = []
    for _, t, y in records:
        n = min(len(t), len(y))
        order = np.argsort(t[:n])
        traces.append(np.interp(common_t, t[:n][order], y[:n][order], left=np.nan, right=np.nan))
    return common_t, np.vstack(traces) if traces else None

fig, axes = plt.subplots(2, 4, figsize=(10.0, 4.8), sharex=True)
for r, region in enumerate(REGIONS):
    for c, day in enumerate(DAYS):
        for e, event_type in enumerate(("Tone", "Pre_PE")):
            ax = axes[r, c * 2 + e]
            t, mat = event_matrix_by_trial(rt_exps[(day, region)].da_df, region, event_type=event_type, time_window=learning_tone_window)
            if mat is None:
                ax.text(0.5, 0.5, "No data", transform=ax.transAxes, ha="center", va="center")
            else:
                im = ax.imshow(mat, aspect="auto", extent=[t[0], t[-1], mat.shape[0], 1], cmap="viridis", vmin=np.nanpercentile(mat, 2), vmax=np.nanpercentile(mat, 98))
                ax.axvline(0, color="white", ls="--", lw=0.8)
                if event_type == "Tone":
                    ax.axvline(4, color="#C2185B", ls="--", lw=0.8)
            ax.set_title(f"{region} {day} {event_type}")
            ax.set_xlabel("Time (s)")
            ax.set_ylabel("Trial")
fig.tight_layout()
plt.show()
save_all(fig, "learning_trial_average_heatmaps_to_20s")

## Single-Animal Day 1 vs Day 10 Heatmaps

In [ ]:
def choose_subject(region):
    if example_subjects.get(region):
        return example_subjects[region]
    subjects = sorted(set(learning_trials.loc[learning_trials["region"] == region, "subject_name"]))
    return subjects[0] if subjects else None

fig, axes = plt.subplots(2, 4, figsize=(10.0, 4.8), sharex=True)
for r, region in enumerate(REGIONS):
    subject = choose_subject(region)
    for c, day in enumerate(DAYS):
        df = rt_exps[(day, region)].da_df.query("subject_name == @subject") if subject else rt_exps[(day, region)].da_df.iloc[0:0]
        for e, event_type in enumerate(("Tone", "Pre_PE")):
            ax = axes[r, c * 2 + e]
            t, mat = event_matrix_by_trial(df, region, event_type=event_type, time_window=learning_tone_window)
            if mat is None:
                ax.text(0.5, 0.5, "No data", transform=ax.transAxes, ha="center", va="center")
            else:
                ax.imshow(mat, aspect="auto", extent=[t[0], t[-1], mat.shape[0], 1], cmap="viridis", vmin=np.nanpercentile(mat, 2), vmax=np.nanpercentile(mat, 98))
                ax.axvline(0, color="white", ls="--", lw=0.8)
                if event_type == "Tone":
                    ax.axvline(4, color="#C2185B", ls="--", lw=0.8)
            ax.set_title(f"{region} {subject} {day} {event_type}")
            ax.set_xlabel("Time (s)")
            ax.set_ylabel("Trial")
fig.tight_layout()
plt.show()
save_all(fig, "learning_single_animal_heatmaps")

## Day 1 vs Day 10 Tone And Port-Entry Statistics

In [ ]:
stat_trials = learning_trials.loc[learning_trials["reward_by_20s"]].copy() if require_reward_pickup else learning_trials.copy()
stat_rows = []
fig, axes = plt.subplots(2, 2, figsize=(5.8, 4.8), sharey=False)
for r, region in enumerate(REGIONS):
    for c, metric in enumerate((tone_metric_col, pe_metric_col)):
        subj = subject_means(stat_trials.query("region == @region"), metric, ["region", "day"])
        groups = [{"label": day, "values": subj.query("day == @day")[metric].to_numpy(), "color": rsf.DAY_COLORS[region][day]} for day in DAYS]
        stats = rsf.plot_bar_groups(axes[r, c], groups, "Mean z-scored dF/F", f"{region} {metric}", comparisons=[(0, 1)])
        stats.insert(0, "region", region)
        stats.insert(1, "metric", metric)
        stat_rows.append(stats)
fig.tight_layout()
plt.show()
save_all(fig, "learning_day1_vs_day10_stat_bars_reward_pickup_trials")
learning_da_stats = pd.concat(stat_rows, ignore_index=True)
learning_da_stats

# Reward Competition Behavior

In [ ]:
rc_trials = trial_rows_from_df(rc_exp.da_df, day="RC", region="both")
winner_col = first_matching_col(rc_exp.da_df, ["filtered_winner_array", "winner_array"])
hvl_comp_col = first_matching_col(rc_exp.da_df, ["HVL_Comp", "hvl_comp"])
hvl_pre_col = first_matching_col(rc_exp.da_df, ["HVL_PreComp", "hvl_precomp"])
print("winner_col", winner_col, "hvl_comp_col", hvl_comp_col, "hvl_pre_col", hvl_pre_col)
rc_exp.da_df[["subject_name", "file name"] + [c for c in (winner_col, hvl_comp_col, hvl_pre_col) if c]].head()

## Percent Trials Won By Overall Session Winner vs Loser

In [ ]:
def session_win_summary(df):
    rows = []
    for _, row in df.iterrows():
        subject = row.get("subject_name", "")
        winners = [w for w in as_list(row.get(winner_col, [])) if not pd.isna(w) and str(w).lower() != "tangle"]
        if not winners:
            continue
        wins = sum(w == subject for w in winners)
        losses = sum(w != subject for w in winners)
        total = wins + losses
        rows.append({"session_id": shared_session_id(row.get("file name", "")), "file name": row.get("file name", ""), "subject_name": subject, "wins": wins, "losses": losses, "total": total, "percent_won": 100 * wins / total if total else np.nan})
    per_mouse = pd.DataFrame(rows)
    if per_mouse.empty:
        return per_mouse, per_mouse
    per_mouse["session_role"] = per_mouse.groupby("session_id")["percent_won"].rank(method="first", ascending=False).map({1.0: "Overall winner", 2.0: "Overall loser"})
    return per_mouse, per_mouse.dropna(subset=["session_role"])

session_win_by_mouse, session_roles = session_win_summary(rc_exp.da_df)
fig, ax = plt.subplots(figsize=(3.0, 2.5))
groups = [
    {"label": "Overall winner", "values": session_roles.query("session_role == 'Overall winner'")["percent_won"].to_numpy(), "color": rsf.OUTCOME_COLORS["Win"]},
    {"label": "Overall loser", "values": session_roles.query("session_role == 'Overall loser'")["percent_won"].to_numpy(), "color": rsf.OUTCOME_COLORS["Loss"]},
]
win_role_stats = rsf.plot_bar_groups(ax, groups, "% trials won", "Session outcome split", comparisons=[(0, 1)])
plt.show()
save_all(fig, "competition_percent_trials_won_overall_winner_loser")
display(session_roles.head())
win_role_stats

## High-Comp vs Low-Comp Trial Split

In [ ]:
def hvl_trial_table(df, comp_col):
    rows = []
    for _, row in df.iterrows():
        labels = as_list(row.get(comp_col, []))
        winners = as_list(row.get(winner_col, [])) if winner_col else []
        for i, label in enumerate(labels):
            try:
                comp = int(float(label))
            except (TypeError, ValueError):
                continue
            winner = winners[i] if i < len(winners) else np.nan
            rows.append({"session_id": shared_session_id(row.get("file name", "")), "subject_name": row.get("subject_name", ""), "trial_idx": i, "comp_label": "High" if comp == 1 else "Low", "outcome": "Win" if winner == row.get("subject_name", "") else "Loss" if not pd.isna(winner) else "Tie"})
    return pd.DataFrame(rows)

hvl_trials = hvl_trial_table(rc_exp.da_df, hvl_comp_col)
hvl_session = hvl_trials.groupby(["session_id", "comp_label"], as_index=False).size()
hvl_session["percent"] = hvl_session.groupby("session_id")["size"].transform(lambda x: 100 * x / x.sum())
fig, ax = plt.subplots(figsize=(2.8, 2.5))
groups = [{"label": label, "values": hvl_session.query("comp_label == @label")["percent"].to_numpy(), "color": rsf.COMP_COLORS[label]} for label in ("Low", "High")]
hvl_stats = rsf.plot_bar_groups(ax, groups, "% trials", "Competition intensity split", comparisons=[(0, 1)])
plt.show()
save_all(fig, "competition_high_low_trial_split")
hvl_stats

## Competitive Bouts Per Session

In [ ]:
session_bouts = hvl_trials.query("comp_label == 'High'").groupby("session_id", as_index=False).size().rename(columns={"size": "high_comp_trials"})
fig, ax = plt.subplots(figsize=(3.0, 2.4))
ax.hist(session_bouts["high_comp_trials"], bins="auto", color=rsf.COMP_COLORS["High"], edgecolor="black", linewidth=0.6)
ax.set_xlabel("High-comp trials per session")
ax.set_ylabel("Sessions")
ax.set_title("Competitive trial distribution")
rsf.style_axis(ax)
plt.show()
save_all(fig, "competition_total_high_comp_trials_histogram")
session_bouts.describe()

## Temporal Dynamics Of Competitive Behavior

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6.4, 2.5))
try:
    rc_exp.plot_session_trial_competitive_bout_metric(ax=axes[0], event_col=hvl_comp_col, metric="frequency_per_min", color=rsf.COMP_COLORS["High"], title="Across session")
except Exception as exc:
    axes[0].text(0.5, 0.5, str(exc), ha="center", va="center", transform=axes[0].transAxes, wrap=True)
try:
    rc_exp.plot_binned_competitive_bout_activity(ax=axes[1], comp_col=hvl_comp_col, trace_color=rsf.COMP_COLORS["High"], title="Around tone")
except Exception as exc:
    axes[1].text(0.5, 0.5, str(exc), ha="center", va="center", transform=axes[1].transAxes, wrap=True)
fig.tight_layout()
plt.show()
save_all(fig, "competition_temporal_dynamics")

## Bout Duration And ITI Between Bouts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6.0, 2.5))
if competition_bout_csv_folder.exists():
    raw_bouts, merged_bouts = rc_exp.load_competition_bout_csv_folder(str(competition_bout_csv_folder))
    axes[0].hist(merged_bouts["merged_bout_duration_s"], bins="auto", color=rsf.COMP_COLORS["High"], edgecolor="black", linewidth=0.6)
    iti_rows = []
    for session_id, g in merged_bouts.sort_values("merged_bout_start_s").groupby("shared_session_id"):
        iti_rows.extend(np.diff(g["merged_bout_start_s"].to_numpy()))
    axes[1].hist(iti_rows, bins="auto", color=rsf.COMP_COLORS["Low"], edgecolor="black", linewidth=0.6)
else:
    for ax in axes:
        ax.text(0.5, 0.5, f"CSV folder not found:\n{competition_bout_csv_folder}", ha="center", va="center", transform=ax.transAxes, wrap=True)
axes[0].set_title("Bout duration")
axes[0].set_xlabel("Duration (s)")
axes[0].set_ylabel("Bouts")
axes[1].set_title("Inter-bout interval")
axes[1].set_xlabel("ITI (s)")
axes[1].set_ylabel("Intervals")
for ax in axes:
    rsf.style_axis(ax)
fig.tight_layout()
plt.show()
save_all(fig, "competition_bout_duration_and_iti")